In [1]:
# Repo-root bootstrap: find the directory containing the `llm_perf` package
# (walking up from CWD) and chdir there so paths like "llm_perf/database/..."
# and "assets/..." resolve regardless of where the notebook lives on disk.
import os
from pathlib import Path
_p = Path.cwd()
while not (_p / "llm_perf" / "__init__.py").exists():
    if _p == _p.parent:
        raise RuntimeError("Could not find the llm_perf repository root above the current directory")
    _p = _p.parent
os.chdir(_p)
print("Working directory: repository root (the directory that contains llm_perf/)")

Working directory: repository root (the directory that contains llm_perf/)


# Quickstart Tutorial
This tutorial walks through the end-to-end `llm_perf` workflow: configuring the environment, loading specs (model/system/partition/tuner), optionally converting a HuggingFace config, and running the `InferenceCalculator` to inspect latency, memory, and communication diagnostics.

Everything the notebook computes comes from files tracked in the repository, so a fresh clone reproduces the same numbers. The one optional step (the HuggingFace conversion in section 2) needs a file the repository does not ship; it is skipped when the file is absent and nothing later depends on it.

## 1. Environment Setup and Imports
We verify the local environment, import the public `llm_perf` APIs, and define a small helper for displaying dataclasses and result bundles as collapsible JSON.

In [2]:
from __future__ import annotations

import math
import platform
import tempfile
from dataclasses import asdict, is_dataclass
from pathlib import Path

from IPython.display import JSON, display

from llm_perf import InferenceCalculator, PartitionSpec
from llm_perf.io import (
    list_hw_system_ids,
    list_model_ids,
    list_tuner_ids,
    load_model_from_db,
    load_model_spec,
    load_system_from_db,
    load_tuner_from_db,
)

# Database IDs are file-name stems under llm_perf/database/<type>/.
MODEL_ID = "example.model.dense"     # small dense model card, inspected in section 2
RUN_MODEL_ID = "qwen3_vl_235b_fp8"   # tracked MoE model card that sections 4-6 partition and run
SYSTEM_ID = "example.sys"
TUNER_ID = "example.tuner"


def json_safe(obj):
    """Copy `obj` with non-finite floats replaced by strings ("inf", "nan").

    JSON has no Infinity or NaN, and some result fields are legitimately
    infinite (for example `B_star`, the batch size at which a step turns
    compute-bound, when the step is memory-bound at every batch size), so
    they are shown as text instead.
    """
    if isinstance(obj, float) and not math.isfinite(obj):
        return str(obj)
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(v) for v in obj]
    return obj


def show_json(obj):
    """Display a dataclass or a plain dict / list as collapsible JSON."""
    data = asdict(obj) if is_dataclass(obj) else obj
    display(JSON(json_safe(data), expanded=False))


print(f"Python {platform.python_version()} running on {platform.platform()}")

print("\nAvailable Database IDs\n=======================")
print("Models:", list_model_ids())
print("Systems:", list_hw_system_ids())
print("Tuners:", list_tuner_ids())

Python 3.13.1 running on macOS-26.6.2-arm64-arm-64bit-Mach-O

Available Database IDs
Models: ['deepseek_r1_0528', 'deepseek_v4_pro', 'example.model.dense', 'glm5', 'gpt_1_8t_moe', 'gpt_oss_120b', 'kimi_k25', 'llama3.1_70b', 'llama3.1_8b', 'minimax_m25', 'qwen35_397b_a17b', 'qwen3_vl_235b_fp8']
Systems: ['b200.8gpu', 'b200.multibox', 'b300.8gpu', 'b300.multibox', 'dmatrix.server', 'dmatrix.squadrack', 'example.sys', 'gb200.72gpu', 'gb200.multibox', 'gb300.72gpu', 'h100.8gpu', 'h100.multibox', 'h200.8gpu', 'h200.multibox', 'tpu.v5p.pod']
Tuners: ['example.tuner', 'gpt_1_8t_moe.tuner', 'large_dense.tuner', 'qwen3.tuner']


## 2. Load and Inspect Model Cards (Optional HF Conversion)
We load two ready-to-use model specs from the on-disk database: the small dense example card, and the Mixture-of-Experts (MoE) card `qwen3_vl_235b_fp8` that the rest of the tutorial partitions and runs. Both are tracked in the repository. An optional cell then shows how to adapt a HuggingFace config into the same schema.

In [3]:
example_spec = load_model_from_db(MODEL_ID)
print(f"\nModel Spec: {MODEL_ID}\n" + "=" * (12 + len(MODEL_ID)))
show_json(example_spec)

# The model the rest of the tutorial runs. It is loaded from the database here,
# not in the optional cell below, so later results do not depend on that cell.
model_spec = load_model_from_db(RUN_MODEL_ID)
print(f"\nModel Spec: {RUN_MODEL_ID}\n" + "=" * (12 + len(RUN_MODEL_ID)))
show_json(model_spec)


Model Spec: example.model.dense


<IPython.core.display.JSON object>


Model Spec: qwen3_vl_235b_fp8


<IPython.core.display.JSON object>

**Optional: convert a HuggingFace config into an `llm_perf` model schema.**
The next cell is a demonstration and nothing later in the notebook depends on it. It needs a HuggingFace `config.json`, which the repository does not ship: download one from a model's page on the HuggingFace Hub and point `HF_SAMPLE` at it. If the file is absent the cell says so and converts nothing. The demonstration writes the adapted card to a temporary directory, so it never changes the database; to keep an adapted card, write it to `llm_perf/database/model/<id>.json` instead.

In [4]:
from llm_perf.utils import convert_hf_config_to_model_json

# Optional demonstration. HF_SAMPLE is a path relative to the repository root;
# the file is not shipped with the repository.
HF_SAMPLE = Path("archives") / "hf" / "qwen3_vl_235b_a22b_thinking_fp8.json"
HF_ADAPTED_NAME = "qwen3_vl_235b_fp8"

converted_spec = None
if HF_SAMPLE.is_file():
    with tempfile.TemporaryDirectory() as tmp_dir:
        converted_path = convert_hf_config_to_model_json(
            hf_config_path=HF_SAMPLE,
            out_path=Path(tmp_dir) / f"{HF_ADAPTED_NAME}.json",
            name_override=HF_ADAPTED_NAME,
            overwrite=True,
        )
        converted_spec = load_model_spec(converted_path)
    print(f"Adapted {HF_SAMPLE.as_posix()} -> {converted_path.name} (in a temporary directory)")
    print("\nConverted Model Spec\n====================")
    show_json(converted_spec)
    if HF_ADAPTED_NAME in list_model_ids():
        same = converted_spec == load_model_from_db(HF_ADAPTED_NAME)
        print(f"Identical to the database entry '{HF_ADAPTED_NAME}': {same}")
    print("An adapted card written to llm_perf/database/model/<id>.json loads like any other database entry.")
else:
    print(f"Skipped: no HuggingFace config at {HF_SAMPLE.as_posix()} (the repository does not ship one).")
    print("Nothing was converted. The rest of the notebook does not use this cell.")

Adapted archives/hf/qwen3_vl_235b_a22b_thinking_fp8.json -> qwen3_vl_235b_fp8.json (in a temporary directory)

Converted Model Spec


<IPython.core.display.JSON object>

Identical to the database entry 'qwen3_vl_235b_fp8': True
An adapted card written to llm_perf/database/model/<id>.json loads like any other database entry.


## 3. Collect and Display Hardware System Info
Next we inspect the hardware/network description used by the simulator by loading the baked JSON spec from the database.

In [5]:
system_spec = load_system_from_db(SYSTEM_ID)
print("\nHardware / Network Spec (from database)\n========================================")
show_json(system_spec)


Hardware / Network Spec (from database)


<IPython.core.display.JSON object>

## 4. Define and Visualize Partition Strategy
`PartitionSpec` controls data/pipeline/tensor/expert/sequence sharding. We inspect the JSON-backed plan and plot the scale of each axis to spot imbalances quickly.

In [6]:
partition_spec = PartitionSpec(PP=1, TP=4, EP=8, SP=2)
print("\nPartition Spec\n==============")
show_json(partition_spec)

dims = {
    "PP": partition_spec.PP,
    "EP": partition_spec.EP,
    "TP": partition_spec.TP,
    "SP": partition_spec.SP,
}

print("\nParallelism Factors\n====================")
for axis, value in dims.items():
    bar = "#" * min(value, 40)
    print(f"{axis}: {value:>4} | {bar}")

print("\nDevice Budget Cross-Check\n==========================")
replica_size = (
    partition_spec.PP
    * max(1, partition_spec.EP)
    * partition_spec.TP
    * max(1, partition_spec.SP)
)
dp = system_spec.num_devices // replica_size
total_devices = dp * replica_size

print(
    "Nested replicas = DP({dp}) → PP({pp}) → EP({ep}) → TP({tp}) → SP({sp}) = {total} devices".format(
        dp=dp,
        pp=partition_spec.PP,
        ep=partition_spec.EP,
        tp=partition_spec.TP,
        sp=partition_spec.SP,
        total=total_devices,
    )
)
print("Ensure this nested count does not exceed `system_spec.num_devices`.")


Partition Spec


<IPython.core.display.JSON object>


Parallelism Factors
PP:    1 | #
EP:    8 | ########
TP:    4 | ####
SP:    2 | ##

Device Budget Cross-Check
Nested replicas = DP(1) → PP(1) → EP(8) → TP(4) → SP(2) = 64 devices
Ensure this nested count does not exceed `system_spec.num_devices`.


## 5. Configure and Display Tuner Parameters
Tuning knobs capture algorithm choices, overlap heuristics, and search-related metadata. We load them, surface the raw values, and log a few derived quantities for reproducibility.

In [7]:
tuner_spec = load_tuner_from_db(TUNER_ID)
print("\nTuning Spec\n===========")
show_json(tuner_spec)

# Phase E: collective counts, algorithm choices, and comm/compute overlap (ρ)
# moved off TuningSpec onto FrameworkSpec. The display below pulls those
# notes from FrameworkSpec.default() — load a per-stack JSON via
# `load_framework_from_db("dynamo_trt")` to inspect a real stack.
from llm_perf.specs.framework_spec import FrameworkSpec
framework_spec = FrameworkSpec.default()

derived = {
    "S_decode_tokens": tuner_spec.S_decode,
    "collectives_per_layer": {
        "TP": framework_spec.n_TP_collectives,
        "EP": framework_spec.n_EP_collectives,
        "SP": framework_spec.n_SP_collectives,
    },
    "comm_overlap_factor": framework_spec.comm_overlap_factor,
    "tp_algorithm_decode": framework_spec.tp_algorithm_decode,
    "ep_algorithm_decode": framework_spec.ep_algorithm_decode,
    "inc_enabled": framework_spec.inc_enabled,
}

print("\nDerived framework notes\n========================")
show_json(derived)


Tuning Spec


<IPython.core.display.JSON object>


Derived framework notes


<IPython.core.display.JSON object>

## 6. Execute Inference and Surface Diagnostics
With all specs in hand we can instantiate `InferenceCalculator`, run the analytical models, and report memory, FLOPs, traffic, communication, and latency breakdowns. These metrics let you validate hardware fit and throughput before spinning up a real cluster.

In [8]:
calculator = InferenceCalculator(model_spec, system_spec, partition_spec, tuner_spec)
results = calculator.run()

print("\nMemory (per device)\n====================")
print(f"Params: {results.memory.M_theta_device / 1e9:.3f} GB")
print(f"Activations: {results.memory.M_act_device / 1e9:.3f} GB")
print(f"KV cache: {results.memory.M_kv_device / 1e9:.3f} GB")
print(f"Total: {results.memory.M_total_device / 1e9:.3f} GB (fits HBM? {results.memory.fits_in_HBM})")

print("\nFLOPs\n=====")
print(f"Per token per device: {results.flops.F_token_device / 1e9:.3f} GFLOPs")

print("\nTraffic\n=======")
print(f"Per token traffic: {results.traffic.T_token_eff / 1e9:.3f} GB")

print("\nCommunication\n=============")
print(f"TP time: {results.comm.t_TP * 1e6:.3f} us")
print(f"EP time: {results.comm.t_EP * 1e6:.3f} us")
print(f"SP time: {results.comm.t_SP * 1e6:.3f} us")
print(f"PP time: {results.comm.t_PP * 1e6:.3f} us")
print(f"Stage comm: {results.comm.t_comm_stage * 1e6:.3f} us")

print("\nLatency / Throughput\n=====================")
print(f"t_compute: {results.latency.t_compute * 1e6:.3f} us")
print(f"t_mem: {results.latency.t_mem * 1e6:.3f} us")
print(f"t_local: {results.latency.t_local * 1e6:.3f} us")
print(f"t_comm: {results.latency.t_comm * 1e6:.3f} us")
print(f"t_step_user: {results.latency.t_step_user * 1e6:.3f} us")
print(f"Single replica TPS: {results.latency.TPS_single:.3f}")
print(f"Total TPS (DP-scaled): {results.latency.TTPS:.3f}")

summary_bundle = {
    "memory": asdict(results.memory),
    "flops": asdict(results.flops),
    "traffic": asdict(results.traffic),
    "comm": asdict(results.comm),
    "latency": asdict(results.latency),
}
# show_json prints non-finite values as text (B_star is inf when the step is
# memory-bound at every batch size): JSON has no Infinity.
print("\nStructured results (JSON)\n=========================")
show_json(summary_bundle)


Memory (per device)
Params: 9.084 GB
Activations: 0.000 GB
KV cache: 0.025 GB
Total: 9.108 GB (fits HBM? True)

FLOPs
=====
Per token per device: 5.125 GFLOPs

Traffic
Per token traffic: 2.144 GB

Communication
TP time: 6.015 us
EP time: 7.072 us
SP time: 2.003 us
PP time: 0.000 us
Stage comm: 2648.604 us

Latency / Throughput
t_compute: 2.563 us
t_mem: 639.944 us
t_local: 639.944 us
t_comm: 2648.604 us
t_step_user: 3335.036 us
Single replica TPS: 299.847
Total TPS (DP-scaled): 299.847

Structured results (JSON)


<IPython.core.display.JSON object>